In [19]:
import pandas as pd
import tensorflow as tf

df = pd.read_csv("train.csv")
df["path"] = "boneage-training-dataset/" + df["id"].astype(str) + ".png"

df_val = pd.read_csv("Validation Dataset.csv")
df_val["path"] = "boneage-validation-dataset/" + df_val["id"].astype(str) + ".png"


In [20]:
from sklearn.preprocessing import StandardScaler

In [21]:
paths = df["path"].values
labels = df[["boneage"]].values.astype("float32")

scaler = StandardScaler()
scaler.fit(labels)
labels_scaled = scaler.transform(labels)

dataset = tf.data.Dataset.from_tensor_slices((paths, labels_scaled))


In [22]:
paths_val = df_val["path"].values
labels_val = df_val[["boneage"]].values.astype("float32")
labels_val_scaled = scaler.transform(labels_val)

dataset_val = tf.data.Dataset.from_tensor_slices((paths_val, labels_val_scaled))

In [23]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)   # grayscale
    img = tf.image.resize(img, (224, 224))
    img = img / 255.0
    return img, label


In [ ]:
dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(32).prefetch(tf.data.AUTOTUNE).shuffle(2000)

dataset_val = dataset_val.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset_val = dataset_val.batch(32).prefetch(tf.data.AUTOTUNE)

In [25]:
# define a simple CNN for regression (e.g. bone age)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 1)),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1)  # regression output (change as needed)
])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='mse',
              metrics=['mae'])

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 224, 224, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,313 (427.00 KB)

 Trainable params: 109,313 (427.00 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


In [27]:
history = model.fit(
    dataset,
    validation_data=dataset_val,
    epochs=50,
    callbacks=[early_stop]
)


Epoch 1/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 168s 421ms/step - loss: 0.9915 - mae: 0.8112 - val_loss: 1.0159 - val_mae: 0.8268
Epoch 2/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 167s 421ms/step - loss: 0.9725 - mae: 0.8003 - val_loss: 0.9526 - val_mae: 0.7838
Epoch 3/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 175s 442ms/step - loss: 0.8701 - mae: 0.7501 - val_loss: 0.8566 - val_mae: 0.7357
Epoch 4/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 177s 447ms/step - loss: 0.8197 - mae: 0.7211 - val_loss: 0.8205 - val_mae: 0.7116
Epoch 5/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 173s 436ms/step - loss: 0.7886 - mae: 0.7029 - val_loss: 0.8459 - val_mae: 0.7275
Epoch 6/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 165s 418ms/step - loss: 0.8065 - mae: 0.7136 - val_loss: 0.7999 - val_mae: 0.6907
Epoch 7/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 164s 414ms/step - loss: 0.7345 - mae: 0.6716 - val_loss: 0.8130 - val_mae: 0.7189
Epoch 8/50
395/395 ━━━━━━━━━━━━━━━━━━━━ 168s 425ms/step - loss: 0.6978 - mae: 0.6539 - val_loss: 0.7387 - val_mae: 0.6624
Epoch 9/50
395/395 ━━━━━

In [28]:
model.save("modello.keras")


In [ ]:
model = tf.keras.models.load_model("modello.keras")
